# 01 — Dataset audit
**Project:** Next-day PM2.5 forecasting for Astana, Kazakhstan (OpenWeatherMap Air Pollution API)

This notebook loads the *actual* downloaded raw dataset, checks its provenance metadata, schema, timestamps,
completeness, duplicates and missing values. Nothing here is simulated: if the raw file does not exist the
notebook stops and tells you which command to run.

Data products: `data/raw/openweather_astana_2021_2025.csv` (raw hourly), `data/raw/dataset_metadata.json`,
`data/raw/chunks/manifest.json` (download log).

In [1]:
# --- Setup: locate the project root and import the package --------------------
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd
from IPython.display import Image, display, Markdown
from air_quality.config import load_config, setup_logging
cfg = load_config(ROOT / "config.yaml"); setup_logging(cfg, "notebooks")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
FIG = cfg.path("figures_dir"); MET = cfg.path("metrics_dir")
print("Project root:", ROOT)

Project root: /Users/dannieey/Desktop/ML_final


In [2]:
if not cfg.path("raw_csv").exists():
    raise FileNotFoundError("Raw data not found. Run: python -m air_quality.cli download  (requires OPENWEATHER_API_KEY in .env)")

## 1. Provenance metadata
The metadata file is written by the downloader and records the endpoint, coordinates, requested interval, number of API requests and record counts.

In [3]:
meta = json.loads(cfg.path("raw_metadata").read_text(encoding="utf-8"))
print(json.dumps({k: meta[k] for k in ["provider", "endpoint", "location", "api_returned_coordinates",
                                       "requested_interval_utc", "n_records", "first_timestamp_utc",
                                       "last_timestamp_utc", "units", "note"]}, indent=2))
print("\nDownload summary:", json.dumps(meta.get("download"), indent=2, default=str))

{
  "provider": "OpenWeatherMap Air Pollution API",
  "endpoint": "https://api.openweathermap.org/data/2.5/air_pollution/history",
  "location": {
    "name": "Astana, Kazakhstan",
    "latitude": 51.1694,
    "longitude": 71.4491,
    "timezone": "Asia/Almaty"
  },
  "api_returned_coordinates": {
    "lat": 51.1694,
    "lon": 71.4491
  },
  "requested_interval_utc": [
    "2020-12-25T18:00:00Z",
    "2025-12-31T19:00:00Z"
  ],
  "n_records": 43273,
  "first_timestamp_utc": "2020-12-25T18:00:00Z",
  "last_timestamp_utc": "2025-12-31T18:00:00Z",
  "units": {
    "co": "ug/m3",
    "no": "ug/m3",
    "no2": "ug/m3",
    "o3": "ug/m3",
    "so2": "ug/m3",
    "pm2_5": "ug/m3",
    "pm10": "ug/m3",
    "nh3": "ug/m3",
    "aqi": "index 1 (Good) - 5 (Very Poor)"
  },
  "note": "Values are API-provided (modelled/aggregated) concentrations for the requested coordinate, not direct measurements from a specific ground station."
}

Download summary: {
  "n_chunks_planned": 262,
  "n_chunks_downl

## 2. Download manifest
Each chunk is recorded as `ok`, `split` (re-requested in smaller pieces) or `failed`. Failed chunks are never silently ignored.

In [4]:
man = json.loads(cfg.path("manifest").read_text(encoding="utf-8"))["chunks"]
mf = pd.DataFrame(man.values())
display(mf["status"].value_counts().rename("chunks").to_frame())
ok = mf[mf.status == "ok"]
print(f"Chunks OK: {len(ok)} | records in OK chunks: {int(ok.n_records.sum())} | "
      f"mean chunk coverage: {100*ok.coverage.mean():.2f}% | chunks below 90% coverage: {(ok.coverage < 0.9).sum()}")
display(ok.sort_values("coverage").head(10)[["start_utc", "end_utc", "n_records", "expected", "coverage"]])

,chunks
status,
ok,265
split,3


Chunks OK: 265 | records in OK chunks: 43273 | mean chunk coverage: 97.63% | chunks below 90% coverage: 24


,start_utc,end_utc,n_records,expected,coverage
227,2025-04-01T06:00:00Z,2025-04-03T00:00:00Z,0.0,42,0.0000
225,2025-03-30T12:00:00Z,2025-04-01T06:00:00Z,0.0,42,0.0000
228,2025-04-03T00:00:00Z,2025-04-04T18:00:00Z,17.0,42,0.4048
81,2022-07-15T18:00:00Z,2022-07-22T18:00:00Z,120.0,168,0.7143
144,2023-09-29T18:00:00Z,2023-10-06T18:00:00Z,120.0,168,0.7143
218,2025-02-28T18:00:00Z,2025-03-07T18:00:00Z,120.0,168,0.7143
224,2025-03-28T18:00:00Z,2025-03-30T12:00:00Z,31.0,42,0.7381
112,2023-02-17T18:00:00Z,2023-02-24T18:00:00Z,127.0,168,0.7560
102,2022-12-09T18:00:00Z,2022-12-16T18:00:00Z,137.0,168,0.8155
231,2025-04-18T18:00:00Z,2025-04-25T18:00:00Z,144.0,168,0.8571


## 3. Raw table: structure and data types

In [5]:
raw = pd.read_csv(cfg.path("raw_csv"))
print(raw.shape)
display(raw.head())
display(raw.dtypes.rename("dtype").to_frame())

(43273, 16)


,dt_unix,timestamp_utc,timestamp_local,date_local,latitude,longitude,provider,aqi,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,1608919200,2020-12-25T18:00:00Z,2020-12-26T00:00:00+06:00,2020-12-26,51.1694,71.4491,OpenWeatherMap Air Pollution API (history),1.0,257.02,0.32,7.80,23.96,12.88,3.22,3.37,0.25
1,1608922800,2020-12-25T19:00:00Z,2020-12-26T01:00:00+06:00,2020-12-26,51.1694,71.4491,OpenWeatherMap Air Pollution API (history),1.0,253.68,0.19,6.26,26.11,10.13,2.77,2.88,0.24
2,1608926400,2020-12-25T20:00:00Z,2020-12-26T02:00:00+06:00,2020-12-26,51.1694,71.4491,OpenWeatherMap Air Pollution API (history),1.0,250.34,0.17,5.66,27.90,9.06,2.73,2.82,0.24
3,1608930000,2020-12-25T21:00:00Z,2020-12-26T03:00:00+06:00,2020-12-26,51.1694,71.4491,OpenWeatherMap Air Pollution API (history),1.0,250.34,0.17,5.57,28.97,8.94,2.77,2.85,0.25
4,1608933600,2020-12-25T22:00:00Z,2020-12-26T04:00:00+06:00,2020-12-26,51.1694,71.4491,OpenWeatherMap Air Pollution API (history),1.0,250.34,0.17,5.57,29.33,8.94,2.84,2.91,0.26


,dtype
dt_unix,int64
timestamp_utc,object
timestamp_local,object
date_local,object
latitude,float64
longitude,float64
provider,object
aqi,float64
co,float64
no,float64


## 4. Audit statistics (timestamps, coverage, missingness, PM2.5 range)

In [6]:
from air_quality.data_validation import audit_hourly, format_audit
a = audit_hourly(raw, cfg.tz, min_valid_hours=cfg["forecast"]["target_min_valid_hours"])
print(format_audit(a, "Raw dataset audit"))

=== Raw dataset audit ===
first_timestamp_utc                2020-12-25T18:00:00+00:00
last_timestamp_utc                 2025-12-31T18:00:00+00:00
first_timestamp_local              2020-12-26T00:00:00+06:00
last_timestamp_local               2025-12-31T23:00:00+05:00
n_records                          43273
n_unique_timestamps                43273
n_duplicated_timestamps            0
expected_hourly_intervals          43969
coverage_pct                       98.417
missing_hours                      696
missing_by_field:
    aqi      0
    co       0
    no       0
    no2      0
    o3       0
    so2      0
    pm2_5    0
    pm10     0
    nh3      0
missing_pct_by_field:
    aqi      0.0
    co       0.0
    no       0.0
    no2      0.0
    o3       0.0
    so2      0.0
    pm2_5    0.0
    pm10     0.0
    nh3      0.0
pm2_5_min                          0.0
pm2_5_max                          87.67
n_calendar_days_local              1832
n_valid_days_min_20h               1783
n

/Users/dannieey/Desktop/ML_final/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [7]:
miss = pd.DataFrame({"missing": a["missing_by_field"], "missing_pct": a["missing_pct_by_field"]})
display(miss)
print(f"Interpretation: the raw file spans {a['first_timestamp_local']} to {a['last_timestamp_local']} (local). "
      f"It contains {a['n_records']} records for {a['expected_hourly_intervals']} expected hourly slots, i.e. "
      f"{a['coverage_pct']:.2f}% timestamp coverage ({a['missing_hours']} missing hours). "
      f"{a['n_valid_days_min_20h']} of {a['n_calendar_days_local']} local days have at least 20 hourly records.")

,missing,missing_pct
aqi,0,0.0
co,0,0.0
no,0,0.0
no2,0,0.0
o3,0,0.0
so2,0,0.0
pm2_5,0,0.0
pm10,0,0.0
nh3,0,0.0


Interpretation: the raw file spans 2020-12-26T00:00:00+06:00 to 2025-12-31T23:00:00+05:00 (local). It contains 43273 records for 43969 expected hourly slots, i.e. 98.42% timestamp coverage (696 missing hours). 1783 of 1832 local days have at least 20 hourly records.


## 5. Coverage by year and month

In [8]:
ts = pd.to_datetime(raw["dt_unix"], unit="s", utc=True).dt.tz_convert(cfg.tz)
cov = ts.groupby([ts.dt.year, ts.dt.month]).size().unstack(0)
hours_in_month = pd.DataFrame({y: [pd.Period(f"{y}-{m:02d}").days_in_month * 24 for m in range(1, 13)] for y in cov.columns}, index=range(1, 13))
display((100 * cov / hours_in_month).round(1).rename_axis(index="month", columns="year"))

year,2020,2021,2022,2023,2024,2025
month,,,,,,
1,NaN,96.8,96.8,100.0,96.8,100.0
2,NaN,100.0,96.4,92.9,100.1,100.0
3,NaN,100.0,100.0,96.8,100.0,84.7
4,NaN,100.0,100.0,100.0,96.7,82.5
5,NaN,100.0,100.0,100.0,100.0,100.0
6,NaN,100.0,100.0,100.0,96.7,100.0
7,NaN,100.0,93.5,96.8,96.8,100.0
8,NaN,100.0,100.0,100.0,100.0,100.0
9,NaN,100.0,100.0,97.6,96.7,100.0


## 6. Units and value ranges
According to the official API documentation all `components` are concentrations in µg/m³ and `main.aqi` is the provider's 1–5 index. The table below checks the observed ranges (negative values would be physically impossible).

In [9]:
display(raw[["aqi", "co", "no", "no2", "o3", "so2", "pm2_5", "pm10", "nh3"]].describe().T.round(3))
print("Negative values per field:", a["n_negative_values"])

,count,mean,std,min,25%,50%,75%,max
aqi,43273.0,1.609,0.652,1.00,1.00,2.00,2.00,5.00
co,43273.0,250.112,122.704,81.06,191.93,247.00,290.39,3952.03
no,43273.0,1.573,5.095,0.00,0.01,0.09,0.85,104.61
no2,43273.0,8.218,68.669,-9999.00,3.13,5.83,10.88,104.19
o3,43273.0,54.950,54.481,-9999.00,38.27,56.51,72.96,246.05
so2,43273.0,17.217,19.764,0.00,6.68,12.04,20.98,312.81
pm2_5,43273.0,5.958,7.139,0.00,1.60,3.46,7.38,87.67
pm10,43273.0,7.413,49.166,-9999.00,1.86,4.14,9.24,224.98
nh3,43273.0,0.334,0.464,0.00,0.06,0.18,0.42,11.65


Negative values per field: {'aqi': 0, 'co': 0, 'no': 0, 'no2': 2, 'o3': 1, 'so2': 0, 'pm2_5': 0, 'pm10': 1, 'nh3': 0}


## 7. Notes on data credibility
* The values are **API-provided concentrations for a coordinate** (51.1694 N, 71.4491 E), produced by the
  provider's modelling/aggregation; they are not readings from a specific ground monitor and are not the
  average over the whole city.
* No ground-station comparison was performed in this project, so no claim of physical validation is made.
* All timestamps are Unix seconds (UTC) at the **start** of each hour and converted to Asia/Almaty for daily
  aggregation. Kazakhstan moved to a single UTC+5 zone on 2024-03-01; the IANA database handles this change.